<a href="https://colab.research.google.com/github/Madihasafi/ITC-300-Data-Visualization-/blob/main/Madiha_Safi%2C_Template_of_w5p1_Exploratory_Data_Analysis_(EDA)_Part_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ITC 300: Data Visualization
## Laboratory Work — Exploratory Data Analysis (EDA), Part 03

**Dataset:** `taxes_unprocessed_with_duplicates.csv`  
**Environment:** Google Colab / Jupyter Notebook  
**Main libraries:** pandas, NumPy, Matplotlib

This laboratory translates the lecture content into a complete hands-on EDA workflow using the tax dataset. The examples follow the lecture progression:

**data preprocessing → data transformation → enhanced EDA workflow → end-to-end analytical pipeline**

The dataset contains taxpayer records with the following fields:

- `index` — source row identifier;
- `ID` — taxpayer/record identifier;
- `Refund` — whether a tax refund is recorded (`Yes`/`No`);
- `Marital Status` — taxpayer marital-status category;
- `Income` — annual income;
- `Cheat` — categorical flag (`Yes`/`No`).

> **Important analytical rule:** The file intentionally contains missing values, repeated business profiles, and an extreme income value. Do not remove or replace unusual values automatically. Inspect them first and make each preprocessing decision explicit.

## Learning Outcomes

After completing this laboratory, you should be able to:

1. Explain EDA as a sequence of small, testable analytical decisions.
2. Load and inspect a CSV dataset with pandas.
3. Audit DataFrame shape, columns, data types, missing values, duplicates, and suspicious numeric values.
4. Apply aggregate functions such as `mean()`, `median()`, `min()`, `max()`, and `sum()`.
5. Detect and handle missing values with `isnull()`, `notnull()`, `dropna()`, and `fillna()`.
6. Clean and transform columns with `drop()`, `replace()`, `rename()`, `astype()`, and `pd.to_numeric()`.
7. Select data with `loc[]`, `iloc[]`, row slicing, and column slicing.
8. Build Boolean masks and use `isin()`, `between()`, and `query()`.
9. Normalize a numeric variable using min–max normalization.
10. Sort, select distinct records, create columns, and modify values safely using `loc[]`.
11. Summarize groups using `groupby()`, `agg()`, and split–apply–combine.
12. Manipulate an index using `set_index()` and `reset_index()`.
13. Compare `query()` and `loc[]` and recognize common pandas mistakes.

## Laboratory Roadmap

### Section A — Setup and Data Loading
### Section B — Data Preprocessing
### Section C — Data Selection, Slicing, Filtering, and Normalization
### Section D — Data Transformation
### Section E — Grouping and Split–Apply–Combine
### Section F — Guided Practice and Reference Solutions

# Section A — Setup and Data Loading

## A1. Import the required Python libraries

The lecture introduces NumPy, SciPy, pandas, Matplotlib, and Seaborn. For this laboratory, pandas performs the main tabular operations, NumPy supports numerical calculations, and Matplotlib is used for one final visualization-ready example.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

print('pandas version:', pd.__version__)

pandas version: 2.2.3


## A2. Load `taxes_unprocessed_with_duplicates.csv`

The code first looks for the CSV in the current working directory. In Google Colab, if the file is not already available, an upload prompt is opened.

In [4]:
# Read the csv source file.

df = pd.read_csv("taxes_unprocessed_with_duplicates.csv")

df["Income"] = df["Income"].astype(str).str.replace(",", "")
df["Income"] = pd.to_numeric(df["Income"], errors="coerce")

print("Shape:", df.shape)
print(df.dtypes)
display(df.head())

Shape: (20, 5)
ID                  int64
Refund             object
Marital Status     object
Income            float64
Cheat              object
dtype: object


,ID,Refund,Marital Status,Income,Cheat
0,1,Yes,Single,"125,000.00",No
1,2,No,Married,"100,000.00",No
2,3,No,Single,"70,000.00",No
3,4,Yes,Married,"120,000.00",No
4,5,No,Divorced,"95,000.00",Yes


## A3. Understand the observation and variables

For this laboratory, one row is treated as one source tax record. The columns describe properties of that record. Before cleaning, verify that the table actually follows this structure.

In [13]:
print('\nColumns:')
print(df.columns.tolist())

print("\nFirst 8 records:")
display(df.head(8))



Columns:
['ID', 'Refund', 'Marital Status', 'Income', 'Cheat']

First 8 records:


,ID,Refund,Marital Status,Income,Cheat
0,1,Yes,Single,"125,000.00",No
1,2,No,Married,"100,000.00",No
2,3,No,Single,"70,000.00",No
3,4,Yes,Married,"120,000.00",No
4,5,No,Divorced,"95,000.00",Yes
5,6,No,Married,"60,000.00",No
6,7,Yes,Divorced,"220,000.00",No
7,8,No,Single,"85,000.00",Yes


## A4. Start with an analytical question

EDA becomes more purposeful when preprocessing is linked to a question. We will use this question later in the end-to-end pipeline:

> **Which marital-status groups have the highest average valid income among non-cheating records after analytical duplicates are removed?**

The word **valid** will be defined explicitly later rather than assumed.

In [10]:
question = (
    'Which marital-status groups have the highest average valid income '
    'among non-cheating records after analytical duplicates are removed?'
)
print(question)

Which marital-status groups have the highest average valid income among non-cheating records after analytical duplicates are removed?


# Section B — Data Preprocessing

## B1. DataFrame quality audit before cleaning

Before changing the dataset, inspect:

- number of rows and columns;
- column names;
- data types;
- missing-value counts;
- duplicate counts;
- categorical values;
- numeric distribution.

In [14]:
print('Shape:', df.shape)

print('\nData types:')
print(df.dtypes)

print('\nMissing values:')
print(df.isna().sum())

print('\nExact duplicate rows:')
print(df.duplicated().sum())

print("\nBroad descriptive profile:")
display(df.describe(include="all"))

Shape: (20, 5)

Data types:
ID                  int64
Refund             object
Marital Status     object
Income            float64
Cheat              object
dtype: object

Missing values:
ID                0
Refund            0
Marital Status    2
Income            2
Cheat             0
dtype: int64

Exact duplicate rows:
4

Broad descriptive profile:


,ID,Refund,Marital Status,Income,Cheat
count,20.00,20,18,18.00,20
unique,NaN,2,3,NaN,2
top,NaN,No,Married,NaN,No
freq,NaN,16,8,NaN,15
mean,8.55,NaN,NaN,"151,111.11",NaN
std,4.62,NaN,NaN,"217,015.56",NaN
min,1.00,NaN,NaN,"60,000.00",NaN
25%,4.75,NaN,NaN,"70,000.00",NaN
50%,8.50,NaN,NaN,"92,500.00",NaN
75%,12.25,NaN,NaN,"115,000.00",NaN


### Interpretation of the initial audit

The supplied file contains **16 rows and 6 columns**. Two fields contain missing information:

- one missing `Income` value;
- one missing `Marital Status` value.

The entire rows are not exact duplicates because `index` and `ID` are unique. However, repeated business profiles become visible when identifiers are excluded. This distinction is essential: **duplicate definition depends on the analytical entity.**

## B2. Aggregate functions on `Income`

Aggregate methods summarize many values into one result. The lecture emphasizes `mean()`, `median()`, `min()`, `max()`, and `sum()`.

In [16]:
income_summary = pd.Series({
    'mean': df['Income'].mean(),
    'median': df['Income'].median(),
    'minimum': df['Income'].min(),
    'maximum': df['Income'].max(),
    'sum': df['Income'].sum()
})

income_summary

,0
mean,"151,111.11"
median,"92,500.00"
minimum,"60,000.00"
maximum,"1,000,000.00"
sum,"2,720,000.00"


The mean is much larger than the median because the dataset contains an income of **1,000,000**. This does not automatically mean the value is wrong; it means the analyst should inspect its influence before choosing a summary statistic.

## B3. Detect missing values with `isnull()` / `isna()` and `notnull()` / `notna()`

`isnull()` and `isna()` are equivalent. `notnull()` and `notna()` are their opposites.

In [17]:
# Count missing values
display(df.isnull().sum())

,0
ID,0
Refund,0
Marital Status,2
Income,2
Cheat,0


In [18]:
#Detect missing values
display(df.isnull().head())

,ID,Refund,Marital Status,Income,Cheat
0,False,False,False,False,False
1,False,False,False,False,False
2,False,False,False,False,False
3,False,False,False,False,False
4,False,False,False,False,False


In [20]:
# Detect missing rows/objects
missing_rows = df[df.isnull().any(axis=1)]

display(missing_rows)

,ID,Refund,Marital Status,Income,Cheat
11,12,No,Married,NaN,No
12,13,No,NaN,"70,000.00",No
18,13,No,NaN,"70,000.00",No
19,12,No,Married,NaN,No


In [22]:
# Count non missing values

display(df.notnull().sum())

,0
ID,20
Refund,20
Marital Status,18
Income,18
Cheat,20


In [23]:
# Count non missing values
display(df.notna().sum())

,0
ID,20
Refund,20
Marital Status,18
Income,18
Cheat,20


In [24]:
# Detect non missing rows/objects
display(df[df.notnull().all(axis=1)])

,ID,Refund,Marital Status,Income,Cheat
0,1,Yes,Single,"125,000.00",No
1,2,No,Married,"100,000.00",No
2,3,No,Single,"70,000.00",No
3,4,Yes,Married,"120,000.00",No
4,5,No,Divorced,"95,000.00",Yes
5,6,No,Married,"60,000.00",No
6,7,Yes,Divorced,"220,000.00",No
7,8,No,Single,"85,000.00",Yes
8,9,No,Married,"75,000.00",No
9,10,No,Single,"90,000.00",Yes


## B4. Compare `dropna()` and `fillna()` responsibly

These methods represent different analytical decisions:

- `dropna()` removes incomplete observations;
- `fillna()` retains observations by replacing missing values.

Never fill a missing financial value with zero unless zero is semantically justified. In this example, we demonstrate **median imputation on a copy** only to show the mechanism. The original DataFrame remains unchanged.

In [25]:
df_dropped = df.dropna().copy()
display(df_dropped)

,ID,Refund,Marital Status,Income,Cheat
0,1,Yes,Single,"125,000.00",No
1,2,No,Married,"100,000.00",No
2,3,No,Single,"70,000.00",No
3,4,Yes,Married,"120,000.00",No
4,5,No,Divorced,"95,000.00",Yes
5,6,No,Married,"60,000.00",No
6,7,Yes,Divorced,"220,000.00",No
7,8,No,Single,"85,000.00",Yes
8,9,No,Married,"75,000.00",No
9,10,No,Single,"90,000.00",Yes


In [29]:
# Fill with Constant Value
df_filled = df.copy()
df_filled['Income'] = df_filled['Income'].fillna(df_filled['Income'].median())
display(df_filled)

,ID,Refund,Marital Status,Income,Cheat
0,1,Yes,Single,"125,000.00",No
1,2,No,Married,"100,000.00",No
2,3,No,Single,"70,000.00",No
3,4,Yes,Married,"120,000.00",No
4,5,No,Divorced,"95,000.00",Yes
5,6,No,Married,"60,000.00",No
6,7,Yes,Divorced,"220,000.00",No
7,8,No,Single,"85,000.00",Yes
8,9,No,Married,"75,000.00",No
9,10,No,Single,"90,000.00",Yes


## B5. Remove columns with `drop()`

In the source file the `ID` column is identifier.

This is useful for record tracking but should not be interpreted as continuous measurements.

For some analyses we may remove the `ID` from the analytical feature table.

In [31]:
features_only = df.drop(columns=['ID'])

display(features_only.head())

,Refund,Marital Status,Income,Cheat
0,Yes,Single,"125,000.00",No
1,No,Married,"100,000.00",No
2,No,Single,"70,000.00",No
3,Yes,Married,"120,000.00",No
4,No,Divorced,"95,000.00",Yes


## B6. Detect duplicates — exact rows versus analytical duplicates

`duplicated()` on the full table reports no exact repeated rows because identifiers differ.

If the business definition of duplication is based on the substantive fields `Refund`, `Marital Status`, `Income`, and `Cheat`, repeated profiles become visible.

In [33]:
#Detect identical dyplicate records
exact_duplicates = df[df.duplicated(keep=False)]

display(exact_duplicates)

,ID,Refund,Marital Status,Income,Cheat
2,3,No,Single,"70,000.00",No
6,7,Yes,Divorced,"220,000.00",No
11,12,No,Married,NaN,No
12,13,No,NaN,"70,000.00",No
16,3,No,Single,"70,000.00",No
17,7,Yes,Divorced,"220,000.00",No
18,13,No,NaN,"70,000.00",No
19,12,No,Married,NaN,No


In [34]:
#Detect identical duplicate records
display(df.duplicated(keep=False).sum())

np.int64(8)

In [35]:
# Duplicated records based on substantive attributes
business_cols = ['Refund', 'Marital Status', 'Income', 'Cheat'] # Define business_cols here if not already
display(df[df.duplicated(subset=business_cols, keep=False)])

,ID,Refund,Marital Status,Income,Cheat
2,3,No,Single,"70,000.00",No
4,5,No,Divorced,"95,000.00",Yes
5,6,No,Married,"60,000.00",No
6,7,Yes,Divorced,"220,000.00",No
10,11,No,Divorced,"95,000.00",Yes
11,12,No,Married,NaN,No
12,13,No,NaN,"70,000.00",No
14,15,No,Married,"60,000.00",No
15,16,No,Divorced,"95,000.00",Yes
16,3,No,Single,"70,000.00",No


## B7. Remove analytical duplicates with `drop_duplicates()`

Because the raw IDs are unique, `drop_duplicates()` must be given a meaningful subset when the analytical goal is to remove repeated profiles.

In [37]:
# Removing substantional duplicate records
df_dedup = df.drop_duplicates(subset=business_cols).copy()

df_dedup = df.drop_duplicates(
    subset=business_cols
).copy()

display(df_dedup)

,ID,Refund,Marital Status,Income,Cheat
0,1,Yes,Single,"125,000.00",No
1,2,No,Married,"100,000.00",No
2,3,No,Single,"70,000.00",No
3,4,Yes,Married,"120,000.00",No
4,5,No,Divorced,"95,000.00",Yes
5,6,No,Married,"60,000.00",No
6,7,Yes,Divorced,"220,000.00",No
7,8,No,Single,"85,000.00",Yes
8,9,No,Married,"75,000.00",No
9,10,No,Single,"90,000.00",Yes


## B8. Replace values with `replace()`

`replace()` is useful for standardizing categorical values. Here we convert `Yes`/`No` into binary indicators **in a separate analytical copy** so the original labels remain available.

In [38]:
import numpy as np
#Remove non essential attributes
main_attributes_only = df_dedup.drop('ID', axis=1)

#Create a copy of dataframe to define the step
replaced_values_df = main_attributes_only.copy()

# Calculate mean income
mean_income = replaced_values_df['Income'].mean()

print('Mean income:', mean_income)

# Replace missing Income values with the mean

replaced_values_df
replaced_values_df['Income'] = (
    replaced_values_df['Income'].fillna(mean_income)
)

display(replaced_values_df)

Mean income: 175833.33333333334


,Refund,Marital Status,Income,Cheat
0,Yes,Single,"125,000.00",No
1,No,Married,"100,000.00",No
2,No,Single,"70,000.00",No
3,Yes,Married,"120,000.00",No
4,No,Divorced,"95,000.00",Yes
5,No,Married,"60,000.00",No
6,Yes,Divorced,"220,000.00",No
7,No,Single,"85,000.00",Yes
8,No,Married,"75,000.00",No
9,No,Single,"90,000.00",Yes


In [39]:
# High Income vs Outlier
#Create a copy of dataframe to define the step

high_income = df_dedup.copy()

high_income['high_income'] = (
    high_income['Income'] >= 120000
)

display(high_income)


,ID,Refund,Marital Status,Income,Cheat,high_income
0,1,Yes,Single,"125,000.00",No,True
1,2,No,Married,"100,000.00",No,False
2,3,No,Single,"70,000.00",No,False
3,4,Yes,Married,"120,000.00",No,True
4,5,No,Divorced,"95,000.00",Yes,False
5,6,No,Married,"60,000.00",No,False
6,7,Yes,Divorced,"220,000.00",No,True
7,8,No,Single,"85,000.00",Yes,False
8,9,No,Married,"75,000.00",No,False
9,10,No,Single,"90,000.00",Yes,False


In [40]:
# Create Boolean mask
filtered_df = df[df['Income'] <= 220000].copy()
display(filtered_df.tail(3))

# Drop records where Income is more than 220000


display(filtered_df.tail(3))

,ID,Refund,Marital Status,Income,Cheat
16,3,No,Single,"70,000.00",No
17,7,Yes,Divorced,"220,000.00",No
18,13,No,NaN,"70,000.00",No


,ID,Refund,Marital Status,Income,Cheat
16,3,No,Single,"70,000.00",No
17,7,Yes,Divorced,"220,000.00",No
18,13,No,NaN,"70,000.00",No


In [41]:
# Calculate mean income
mean_income = filtered_df['Income'].mean()

# Replace missing Income values with the mean
# Assuming this refers to 'filtered_df'
filtered_df['Income'] = filtered_df['Income'].fillna(mean_income)
display(filtered_df)



,ID,Refund,Marital Status,Income,Cheat
0,1,Yes,Single,"125,000.00",No
1,2,No,Married,"100,000.00",No
2,3,No,Single,"70,000.00",No
3,4,Yes,Married,"120,000.00",No
4,5,No,Divorced,"95,000.00",Yes
5,6,No,Married,"60,000.00",No
6,7,Yes,Divorced,"220,000.00",No
7,8,No,Single,"85,000.00",Yes
8,9,No,Married,"75,000.00",No
9,10,No,Single,"90,000.00",Yes


## B9. Rename columns with `rename()`

Column names with spaces are valid in pandas, but simple names are often easier to use in expressions and method chains.

In [42]:
df_named=filtered_df.copy()


print(df_named.columns.tolist())
display(df_named.head())

['ID', 'Refund', 'Marital Status', 'Income', 'Cheat']


,ID,Refund,Marital Status,Income,Cheat
0,1,Yes,Single,"125,000.00",No
1,2,No,Married,"100,000.00",No
2,3,No,Single,"70,000.00",No
3,4,Yes,Married,"120,000.00",No
4,5,No,Divorced,"95,000.00",Yes


## B10. Convert data types with `astype()` and `pd.to_numeric()`

Correct types are required before aggregation and visualization. `pd.to_numeric(..., errors='coerce')` safely turns invalid numeric tokens into missing values for later inspection. Categorical fields can be converted to pandas `category` dtype.

In [45]:
df_typed = df_named.copy()
df_typed['Refund'] = df_typed['Refund'].astype('category')
df_typed['Marital Status'] = df_typed['Marital Status'].astype('category')
df_typed['Cheat'] = df_typed['Cheat'].astype('category')


print(df_typed.dtypes)

ID                   int64
Refund            category
Marital Status    category
Income             float64
Cheat             category
dtype: object


# Section C — Data Selection, Slicing, Filtering, and Normalization

## C1. Label-based selection with `loc[]`

`loc[]` selects by labels and is especially useful when you need both row conditions and named columns.

In [46]:
mydata=df_typed.copy()
# Refering Attributes for selection by their Indexes

# When using [], the result is a Pandas DataFrame.
#A list of indexes [[0, 2]] returns a Pandas DataFrame.

display(mydata.iloc[[0, 2]])

,ID,Refund,Marital Status,Income,Cheat
0,1,Yes,Single,"125,000.00",No
2,3,No,Single,"70,000.00",No


In [47]:
# Refering Features for selection by their Names/Labels
display(
    mydata.loc[
        :,
        ['Marital Status', 'Income']
    ]
)

,Marital Status,Income
0,Single,"125,000.00"
1,Married,"100,000.00"
2,Single,"70,000.00"
3,Married,"120,000.00"
4,Divorced,"95,000.00"
5,Married,"60,000.00"
6,Divorced,"220,000.00"
7,Single,"85,000.00"
8,Married,"75,000.00"
9,Single,"90,000.00"


In [48]:
# The series of the first Row
display(mydata.loc[0])
# Single index for one row [1]
# returns a Pandas Series.

,0
ID,1
Refund,Yes
Marital Status,Single
Income,"125,000.00"
Cheat,No


In [50]:
#The series of the first Row

# Single label for one row [1]
display(mydata.iloc[0])

,0
ID,1
Refund,Yes
Marital Status,Single
Income,"125,000.00"
Cheat,No


In [51]:
# The DataFrame of first 5 rows

# In iloc the last index is not included
display(mydata.iloc[0:5])

,ID,Refund,Marital Status,Income,Cheat
0,1,Yes,Single,"125,000.00",No
1,2,No,Married,"100,000.00",No
2,3,No,Single,"70,000.00",No
3,4,Yes,Married,"120,000.00",No
4,5,No,Divorced,"95,000.00",Yes


In [52]:
# The DataFrame of first 6 rows

# In loc the last label is included
display(mydata.loc[0:5])

,ID,Refund,Marital Status,Income,Cheat
0,1,Yes,Single,"125,000.00",No
1,2,No,Married,"100,000.00",No
2,3,No,Single,"70,000.00",No
3,4,Yes,Married,"120,000.00",No
4,5,No,Divorced,"95,000.00",Yes
5,6,No,Married,"60,000.00",No


In [53]:
#The DataFrame of last 5 rows
display(mydata.tail(5))

,ID,Refund,Marital Status,Income,Cheat
14,15,No,Married,"60,000.00",No
15,16,No,Divorced,"95,000.00",Yes
16,3,No,Single,"70,000.00",No
17,7,Yes,Divorced,"220,000.00",No
18,13,No,NaN,"70,000.00",No


In [55]:
# A DataFrame of first six elements
# of "Marital Status" and "Income" Columns

display(
    mydata.loc[
        mydata.index[:6],
        ['Marital Status', 'Income']
    ]
)


,Marital Status,Income
0,Single,"125,000.00"
1,Married,"100,000.00"
2,Single,"70,000.00"
3,Married,"120,000.00"
4,Divorced,"95,000.00"
5,Married,"60,000.00"


In [56]:
# A DataFrame of last six elements
# of 2nd and 4th Columns

display(mydata.iloc[-6:, [1, 3]])

,Refund,Income
12,No,"70,000.00"
14,No,"60,000.00"
15,No,"95,000.00"
16,No,"70,000.00"
17,Yes,"220,000.00"
18,No,"70,000.00"


In [57]:
#A Series of first five elements of "Tax Evasion" Column
display(mydata.loc[0:4, 'Cheat'])

,Cheat
0,No
1,No
2,No
3,No
4,Yes


In [58]:
# Displaying the rows with "income"> 100000

higher_income_data = df[df['Income'] > 100000]
display(higher_income_data)

,ID,Refund,Marital Status,Income,Cheat
0,1,Yes,Single,"125,000.00",No
3,4,Yes,Married,"120,000.00",No
6,7,Yes,Divorced,"220,000.00",No
13,14,No,Married,"1,000,000.00",No
17,7,Yes,Divorced,"220,000.00",No


## C2. Position-based selection with `iloc[]`

`iloc[]` selects using zero-based integer positions. The stop position of a slice is excluded.

In [59]:
# First five rows; columns at positions 1 through 4
subset_iloc = df.iloc[0:5, 1:5]
display(subset_iloc)

,Refund,Marital Status,Income,Cheat
0,Yes,Single,"125,000.00",No
1,No,Married,"100,000.00",No
2,No,Single,"70,000.00",No
3,Yes,Married,"120,000.00",No
4,No,Divorced,"95,000.00",Yes


## C3. Row and column slicing

- `df[0:5]` returns row positions 0 through 4.
- `df[['Refund', 'Income']]` returns selected columns as a DataFrame.

In [60]:
income_over_100k = df['Income'] > 100000
print(f"Number of records with income over 100k: {income_over_100k.sum()}")

Number of records with income over 100k: 5


## C4. Boolean filtering using relational operators

A relational comparison returns one True/False value for each row. When the mask is placed inside `df[...]`, only True rows remain.

In [61]:

display(df[income_over_100k])

,ID,Refund,Marital Status,Income,Cheat
0,1,Yes,Single,"125,000.00",No
3,4,Yes,Married,"120,000.00",No
6,7,Yes,Divorced,"220,000.00",No
13,14,No,Married,"1,000,000.00",No
17,7,Yes,Divorced,"220,000.00",No


## C5. Filter multiple categories with `isin()`

`isin()` is more readable than writing several OR conditions for the same column.

In [62]:
#Display the records with Single or Divorced Marital Status


selected_statuses = df[df['Marital Status'].isin(['Single', 'Divorced'])]
display(selected_statuses)

,ID,Refund,Marital Status,Income,Cheat
0,1,Yes,Single,"125,000.00",No
2,3,No,Single,"70,000.00",No
4,5,No,Divorced,"95,000.00",Yes
6,7,Yes,Divorced,"220,000.00",No
7,8,No,Single,"85,000.00",Yes
9,10,No,Single,"90,000.00",Yes
10,11,No,Divorced,"95,000.00",Yes
15,16,No,Divorced,"95,000.00",Yes
16,3,No,Single,"70,000.00",No
17,7,Yes,Divorced,"220,000.00",No


## C6. Filter an interval with `between()`

`between(75000, 125000)` is inclusive by default, so values equal to either endpoint are retained.

In [63]:
mid_income = df[df['Income'].between(75000, 125000)]
display(mid_income)

,ID,Refund,Marital Status,Income,Cheat
0,1,Yes,Single,"125,000.00",No
1,2,No,Married,"100,000.00",No
3,4,Yes,Married,"120,000.00",No
4,5,No,Divorced,"95,000.00",Yes
7,8,No,Single,"85,000.00",Yes
8,9,No,Married,"75,000.00",No
9,10,No,Single,"90,000.00",Yes
10,11,No,Divorced,"95,000.00",Yes
15,16,No,Divorced,"95,000.00",Yes


## C7. Data normalization — min–max scaling

Min–max normalization maps the observed minimum to 0 and the maximum to 1:

\[
x_{norm}=\frac{x-x_{min}}{x_{max}-x_{min}}
\]

The extreme value 1,000,000 will compress most other normalized incomes toward the lower end. This is a useful demonstration of why normalization does **not** remove outliers.

In [64]:
normalized = df[['ID', 'Income']].copy()

income_min = normalized['Income'].min()
income_max = normalized['Income'].max()
normalized['income_minmax'] = (
    (normalized['Income'] - income_min) /
    (income_max - income_min)
)

print('Original range:', (income_min, income_max))
print('Normalized range:',
      (normalized['income_minmax'].min(),
       normalized['income_minmax'].max()))
display(normalized)

Original range: (60000.0, 1000000.0)
Normalized range: (0.0, 1.0)


,ID,Income,income_minmax
0,1,"125,000.00",0.07
1,2,"100,000.00",0.04
2,3,"70,000.00",0.01
3,4,"120,000.00",0.06
4,5,"95,000.00",0.04
5,6,"60,000.00",0.00
6,7,"220,000.00",0.17
7,8,"85,000.00",0.03
8,9,"75,000.00",0.02
9,10,"90,000.00",0.03


#**Sections forward are for indipendent study**
##The simmilar tasks are going to be implemented on async session but with another dataset

# Section D — Data Transformation

## D1. Boolean masks can also be summarized

In pandas, `True` behaves like 1 and `False` like 0. Therefore, the mean of a Boolean mask is the proportion of rows satisfying the condition.

In [65]:
import pandas as pd
# Read the csv source file.
df = pd.read_csv("taxes_unprocessed_with_duplicates.csv")

under_100k = df['Income'] < 100000

print('Count under 100,000:', under_100k.sum())
print('Proportion under 100,000:', under_100k.mean())

Count under 100,000: 12
Proportion under 100,000: 0.6


## D2. Combine multiple query conditions with `&`

For pandas Series:

- `&` means AND;
- `|` means OR;
- `~` means NOT.

Each comparison should be enclosed in parentheses.

In [66]:
married_mid_income = df[
    (df['Marital Status'] == 'Married') &
    (df['Income'] < 150000)
]

display(married_mid_income)

,ID,Refund,Marital Status,Income,Cheat
1,2,No,Married,"100,000.00",No
3,4,Yes,Married,"120,000.00",No
5,6,No,Married,"60,000.00",No
8,9,No,Married,"75,000.00",No
14,15,No,Married,"60,000.00",No


## D3. `query()` as an alternative filtering syntax

Because `Marital Status` contains a space, it must be surrounded with backticks inside a raw `query()` expression. Renaming the column first avoids this requirement.

In [67]:
query_result = df.query("`Marital Status` == 'Married' & Income < 150000")
display(query_result)

,ID,Refund,Marital Status,Income,Cheat
1,2,No,Married,"100,000.00",No
3,4,Yes,Married,"120,000.00",No
5,6,No,Married,"60,000.00",No
8,9,No,Married,"75,000.00",No
14,15,No,Married,"60,000.00",No


## D4. Filter missing and non-missing values

Missing-value filtering is part of data cleaning and should be linked to the current analytical question.

In [68]:
print('Missing Income:')
display(df[df['Income'].isna()])

print('Non-missing Income:')
display(df[df['Income'].notna()].head())

Missing Income:


,ID,Refund,Marital Status,Income,Cheat
11,12,No,Married,NaN,No
19,12,No,Married,NaN,No


Non-missing Income:


,ID,Refund,Marital Status,Income,Cheat
0,1,Yes,Single,"125,000.00",No
1,2,No,Married,"100,000.00",No
2,3,No,Single,"70,000.00",No
3,4,Yes,Married,"120,000.00",No
4,5,No,Divorced,"95,000.00",Yes


## D5. Sort records with `sort_values()`

Sorting does not change the original DataFrame unless the result is assigned back.

In [69]:
print('Lowest incomes:')
display(df.sort_values('Income').head(8))

print('Highest incomes:')
display(df.sort_values('Income', ascending=False).head(8))

Lowest incomes:


,ID,Refund,Marital Status,Income,Cheat
5,6,No,Married,"60,000.00",No
14,15,No,Married,"60,000.00",No
12,13,No,NaN,"70,000.00",No
2,3,No,Single,"70,000.00",No
16,3,No,Single,"70,000.00",No
18,13,No,NaN,"70,000.00",No
8,9,No,Married,"75,000.00",No
7,8,No,Single,"85,000.00",Yes


Highest incomes:


,ID,Refund,Marital Status,Income,Cheat
13,14,No,Married,"1,000,000.00",No
6,7,Yes,Divorced,"220,000.00",No
17,7,Yes,Divorced,"220,000.00",No
0,1,Yes,Single,"125,000.00",No
3,4,Yes,Married,"120,000.00",No
1,2,No,Married,"100,000.00",No
10,11,No,Divorced,"95,000.00",Yes
15,16,No,Divorced,"95,000.00",Yes


## D6. Select distinct records

Distinct categories can be inspected with `drop_duplicates()` or `unique()`. Distinct business profiles require a deliberate subset of columns.

In [70]:
print('Distinct marital-status values:')
print(df['Marital Status'].drop_duplicates().tolist())

print('\nDistinct Refund/Cheat combinations:')
display(df[['Refund', 'Cheat']].drop_duplicates().reset_index(drop=True))

print('\nDistinct business profiles:', len(df.drop_duplicates(subset=business_cols)))

Distinct marital-status values:
['Single', 'Married', 'Divorced', nan]

Distinct Refund/Cheat combinations:


,Refund,Cheat
0,Yes,No
1,No,No
2,No,Yes



Distinct business profiles: 13


## D7. Create new columns by assignment

A direct assignment writes a new variable into the DataFrame. Work on a copy when the original must remain unchanged.

In [71]:
df_assign = df.copy()
df_assign['high_income'] = df_assign['Income'] >= 120000

display(df_assign[['ID', 'Income', 'high_income']])

,ID,Income,high_income
0,1,"125,000.00",True
1,2,"100,000.00",False
2,3,"70,000.00",False
3,4,"120,000.00",True
4,5,"95,000.00",False
5,6,"60,000.00",False
6,7,"220,000.00",True
7,8,"85,000.00",False
8,9,"75,000.00",False
9,10,"90,000.00",False


## D8. Create new columns with `assign()`

`assign()` is particularly useful in method chains because it returns a new DataFrame.

In [72]:
assigned_view = (
    df[['ID', 'Refund', 'Marital Status', 'Income', 'Cheat']]
      .assign(
          income_k=lambda x: x['Income'] / 1000,
          suspicious=lambda x: (x['Cheat'] == 'Yes') | (x['Income'] >= 500000)
      )
)

display(assigned_view)

,ID,Refund,Marital Status,Income,Cheat,income_k,suspicious
0,1,Yes,Single,"125,000.00",No,125.00,False
1,2,No,Married,"100,000.00",No,100.00,False
2,3,No,Single,"70,000.00",No,70.00,False
3,4,Yes,Married,"120,000.00",No,120.00,False
4,5,No,Divorced,"95,000.00",Yes,95.00,True
5,6,No,Married,"60,000.00",No,60.00,False
6,7,Yes,Divorced,"220,000.00",No,220.00,False
7,8,No,Single,"85,000.00",Yes,85.00,True
8,9,No,Married,"75,000.00",No,75.00,False
9,10,No,Single,"90,000.00",Yes,90.00,True


## D9. Conditional modification with `loc[]`

Use `loc[mask, column]` for safe, explicit assignment. The example replaces the missing marital status with `Unknown` **on a copy**.

In [73]:
df_modified = df.copy()
missing_status = df_modified['Marital Status'].isna()

df_modified.loc[missing_status, 'Marital Status'] = 'Unknown'

display(df_modified.loc[missing_status])

,ID,Refund,Marital Status,Income,Cheat
12,13,No,Unknown,"70,000.00",No
18,13,No,Unknown,"70,000.00",No


## D10. Drop a derived column

`drop(columns=[...])` is explicit and works well in pipelines. `del df['column']` is another option when modifying a DataFrame in place.

In [74]:
temp = df_assign.copy()
print('Before:', temp.columns.tolist())

temp = temp.drop(columns=['high_income'])
print('After:', temp.columns.tolist())

Before: ['ID', 'Refund', 'Marital Status', 'Income', 'Cheat', 'high_income']
After: ['ID', 'Refund', 'Marital Status', 'Income', 'Cheat']


## D11. Descriptive statistics with `describe()`

`describe()` summarizes numeric columns by default. Use `include='all'` when you also want categorical summaries.

In [75]:
print('Numeric description:')
display(df.describe())

print('All-column description:')
display(df.describe(include='all'))

Numeric description:


,ID,Income
count,20.00,18.00
mean,8.55,"151,111.11"
std,4.62,"217,015.56"
min,1.00,"60,000.00"
25%,4.75,"70,000.00"
50%,8.50,"92,500.00"
75%,12.25,"115,000.00"
max,16.00,"1,000,000.00"


All-column description:


,ID,Refund,Marital Status,Income,Cheat
count,20.00,20,18,18.00,20
unique,NaN,2,3,NaN,2
top,NaN,No,Married,NaN,No
freq,NaN,16,8,NaN,15
mean,8.55,NaN,NaN,"151,111.11",NaN
std,4.62,NaN,NaN,"217,015.56",NaN
min,1.00,NaN,NaN,"60,000.00",NaN
25%,4.75,NaN,NaN,"70,000.00",NaN
50%,8.50,NaN,NaN,"92,500.00",NaN
75%,12.25,NaN,NaN,"115,000.00",NaN


## D12. Find the maximum value and the record containing it

First compute the maximum, then use a Boolean condition to retrieve the corresponding row(s).

In [76]:
max_income = df['Income'].max()
print('Maximum Income:', max_income)

display(df[df['Income'] == max_income])

Maximum Income: 1000000.0


,ID,Refund,Marital Status,Income,Cheat
13,14,No,Married,"1,000,000.00",No


## D13. Set, use, and reset an index

`set_index()` changes how rows are labeled. Once `ID` is the index, records can be addressed by ID labels. `reset_index()` restores `ID` as a regular column.

In [77]:
df_indexed = df.copy().set_index('ID')
print('DataFrame indexed by ID:')
display(df_indexed.head())

# Demonstrate label-based deletion on a copy
without_id_14 = df_indexed.drop(index=14)
print('ID 14 present after drop?', 14 in without_id_14.index)

restored = without_id_14.reset_index()
display(restored.head())

DataFrame indexed by ID:


,Refund,Marital Status,Income,Cheat
ID,,,,
1,Yes,Single,"125,000.00",No
2,No,Married,"100,000.00",No
3,No,Single,"70,000.00",No
4,Yes,Married,"120,000.00",No
5,No,Divorced,"95,000.00",Yes


ID 14 present after drop? False


,ID,Refund,Marital Status,Income,Cheat
0,1,Yes,Single,"125,000.00",No
1,2,No,Married,"100,000.00",No
2,3,No,Single,"70,000.00",No
3,4,Yes,Married,"120,000.00",No
4,5,No,Divorced,"95,000.00",Yes


## D14. Recommended row removal — Boolean filtering

Rather than iterating through every row and dropping inside a loop, create a new DataFrame using a Boolean condition.

In [78]:
without_id_14_filter = df[df['ID'] != 14].copy()
print('Original rows:', len(df))
print('Rows after filtering ID 14:', len(without_id_14_filter))

Original rows: 20
Rows after filtering ID 14: 19


# Section E — Grouping and Split–Apply–Combine

## E1. Create a `groupby()` object

`groupby()` implements **split–apply–combine**:

1. **Split** rows according to group keys;
2. **Apply** a function such as mean, sum, or standard deviation;
3. **Combine** the results into a Series or DataFrame.

Creating the group object does not itself produce the final summary.

In [79]:
grouped_by_status = df.groupby('Marital Status')
print(grouped_by_status)

print('\nGroup sizes:')
print(grouped_by_status.size())


Group sizes:
Marital Status
Divorced    5
Married     8
Single      5
dtype: int64


## E2. Calculate mean income by marital status

Selecting one column with single brackets returns a Series after aggregation.

In [80]:
mean_income_by_status = df.groupby('Marital Status')['Income'].mean()
mean_income_by_status

,Income
Marital Status,
Divorced,"145,000.00"
Married,"235,833.33"
Single,"88,000.00"


## E3. Calculate group descriptions

`groupby(...)[column].describe()` gives count, mean, standard deviation, quartiles, and extremes for each group.

In [81]:
group_income_description = df.groupby('Marital Status')['Income'].describe()
display(group_income_description)

,count,mean,std,min,25%,50%,75%,max
Marital Status,,,,,,,,
Divorced,5.00,"145,000.00","68,465.32","95,000.00","95,000.00","95,000.00","220,000.00","220,000.00"
Married,6.00,"235,833.33","375,105.54","60,000.00","63,750.00","87,500.00","115,000.00","1,000,000.00"
Single,5.00,"88,000.00","22,527.76","70,000.00","70,000.00","85,000.00","90,000.00","125,000.00"


## E4. Group-level mean and standard deviation

The lecture uses both built-in aggregation and a lambda expression. Here NumPy's population standard deviation (`np.std`) is applied within each marital-status group.

In [82]:
mean_by_status = df.groupby('Marital Status')['Income'].mean()
std_by_status = df.groupby('Marital Status')['Income'].apply(lambda x: np.std(x.dropna()))

comparison = pd.DataFrame({
    'mean_income': mean_by_status,
    'population_std_income': std_by_status
})

display(comparison)

,mean_income,population_std_income
Marital Status,,
Divorced,"145,000.00","61,237.24"
Married,"235,833.33","342,422.94"
Single,"88,000.00","20,149.44"


## E5. Iterate through groups

Iteration makes the dictionary-like structure of a groupby object visible. It is useful for learning and custom logic, but vectorized aggregation is usually more concise.

In [83]:
for status, group in df.groupby('Marital Status'):
    print(
        'Status:', status,
        '| records:', len(group),
        '| mean income:', round(group['Income'].mean(), 2),
        '| std:', round(group['Income'].std(), 2)
    )

Status: Divorced | records: 5 | mean income: 145000.0 | std: 68465.32
Status: Married | records: 8 | mean income: 235833.33 | std: 375105.54
Status: Single | records: 5 | mean income: 88000.0 | std: 22527.76


## E6. Named group-level aggregation

Named aggregation creates presentation-ready output columns and calculates several measures in one operation.

In [84]:
status_summary = (
    df.groupby('Marital Status')
      .agg(
          taxpayer_records=('ID', 'count'),
          known_income_count=('Income', 'count'),
          total_income=('Income', 'sum'),
          mean_income=('Income', 'mean'),
          median_income=('Income', 'median'),
          refund_yes=('Refund', lambda s: (s == 'Yes').sum()),
          cheat_yes=('Cheat', lambda s: (s == 'Yes').sum())
      )
      .sort_values('mean_income', ascending=False)
)

display(status_summary)

,taxpayer_records,known_income_count,total_income,mean_income,median_income,refund_yes,cheat_yes
Marital Status,,,,,,,
Married,8,6,"1,415,000.00","235,833.33","87,500.00",1,0
Divorced,5,5,"725,000.00","145,000.00","95,000.00",2,3
Single,5,5,"440,000.00","88,000.00","85,000.00",1,2


## E7. Multi-key grouping

Grouping by more than one categorical variable creates a hierarchical result. Here we compare income by marital status and refund flag.

In [85]:
multi_group = (
    df.groupby(['Marital Status', 'Refund'])
      .agg(
          records=('ID', 'count'),
          mean_income=('Income', 'mean')
      )
)

display(multi_group)

records  mean_income
Marital Status Refund                      
Divorced       No            3    95,000.00
               Yes           2   220,000.00
Married        No            7   259,000.00
               Yes           1   120,000.00
Single         No            4    78,750.00
               Yes           1   125,000.00

## E8. `query()` versus `loc[]`

Both expressions below identify non-cheating Married records with known income below 150,000. `query()` is compact; `loc[]` is explicit and simultaneously selects output columns.

In [86]:
query_view = df.query(
    "`Marital Status` == 'Married' & Cheat == 'No' & Income < 150000"
)

loc_view = df.loc[
    (df['Marital Status'] == 'Married') &
    (df['Cheat'] == 'No') &
    (df['Income'] < 150000),
    ['ID', 'Refund', 'Marital Status', 'Income', 'Cheat']
]

print('Same number of rows?', len(query_view) == len(loc_view))
display(loc_view)

Same number of rows? True


,ID,Refund,Marital Status,Income,Cheat
1,2,No,Married,"100,000.00",No
3,4,Yes,Married,"120,000.00",No
5,6,No,Married,"60,000.00",No
8,9,No,Married,"75,000.00",No
14,15,No,Married,"60,000.00",No


## E9. Column selection, assignment, and renaming in one readable chain

This pattern keeps only needed variables, creates derived variables, and gives columns presentation-ready names while leaving the raw DataFrame unchanged.

In [87]:
eda_view = (
    df[['ID', 'Refund', 'Marital Status', 'Income', 'Cheat']]
      .assign(
          income_thousands=lambda x: x['Income'] / 1000,
          high_income=lambda x: x['Income'] >= 120000,
          refund_and_clean=lambda x: (x['Refund'] == 'Yes') & (x['Cheat'] == 'No')
      )
      .rename(columns={
          'Marital Status': 'marital_status',
          'Income': 'annual_income'
      })
)

display(eda_view)

,ID,Refund,marital_status,annual_income,Cheat,income_thousands,high_income,refund_and_clean
0,1,Yes,Single,"125,000.00",No,125.00,True,True
1,2,No,Married,"100,000.00",No,100.00,False,False
2,3,No,Single,"70,000.00",No,70.00,False,False
3,4,Yes,Married,"120,000.00",No,120.00,True,True
4,5,No,Divorced,"95,000.00",Yes,95.00,False,False
5,6,No,Married,"60,000.00",No,60.00,False,False
6,7,Yes,Divorced,"220,000.00",No,220.00,True,True
7,8,No,Single,"85,000.00",Yes,85.00,False,False
8,9,No,Married,"75,000.00",No,75.00,False,False
9,10,No,Single,"90,000.00",Yes,90.00,False,False


## E10. Common EDA mistakes and better practice

| Common mistake | Better practice in this dataset |
|---|---|
| Delete the 1,000,000 income immediately | Inspect its influence and business meaning first |
| Use Python `and` / `or` on Series | Use `&` / `|` with each comparison in parentheses |
| Fill missing `Income` with 0 automatically | Fill only when zero is semantically valid; otherwise drop, impute, or leave missing with justification |
| Treat `ID` or `index` as a measurement | Treat them as identifiers |
| Call the full rows “not duplicated” and stop | Check duplicates on a meaningful business subset |
| Group categories before standardizing labels | Trim and case-normalize labels first |
| Modify `df[mask]['Income']` | Use `df.loc[mask, 'Income']` |
| Plot before validating numeric types | Convert and audit types first |
| Assume `inplace=True` is always clearer or faster | Prefer explicit reassignment when it improves readability |

# Section F — Guided Practice

## F1. Exercises — try these before opening the reference solutions

1. **Missing values:** Count missing values in every column and return only columns with at least one missing value.
2. **Analytical duplicates:** Identify all rows that participate in duplicate business profiles using `Refund`, `Marital Status`, `Income`, and `Cheat`.
3. **Filtering:** Return Single or Divorced taxpayers with income at least 90,000.
4. **Combined conditions:** Return records where `Refund == 'Yes'`, `Cheat == 'No'`, and income is known.
5. **Sorting:** Display the five highest incomes with `ID`, `Marital Status`, `Income`, `Refund`, and `Cheat`.
6. **Assignment:** Create `income_band` with values `Low`, `Medium`, and `High` using thresholds of `< 80,000`, `80,000–119,999`, and `>= 120,000`.
7. **GroupBy:** For each marital-status group, calculate record count, mean income, median income, and count of `Cheat == 'Yes'`.
8. **query():** Use `query()` to return incomes between 75,000 and 125,000 inclusive.
9. **Index manipulation:** Set `ID` as the index, remove the row with ID 14 on a copy, and reset the index.

## F2. Reference Solutions

In [88]:
# Exercise 1
missing_only = df.isna().sum()
missing_only[missing_only > 0]

,0
Marital Status,2
Income,2


In [89]:
# Exercise 2
business_cols = ['Refund', 'Marital Status', 'Income', 'Cheat']
dup_rows = df[df.duplicated(subset=business_cols, keep=False)]
display(dup_rows)

,ID,Refund,Marital Status,Income,Cheat
2,3,No,Single,"70,000.00",No
4,5,No,Divorced,"95,000.00",Yes
5,6,No,Married,"60,000.00",No
6,7,Yes,Divorced,"220,000.00",No
10,11,No,Divorced,"95,000.00",Yes
11,12,No,Married,NaN,No
12,13,No,NaN,"70,000.00",No
14,15,No,Married,"60,000.00",No
15,16,No,Divorced,"95,000.00",Yes
16,3,No,Single,"70,000.00",No


In [90]:
# Exercise 3
ex3 = df[
    df['Marital Status'].isin(['Single', 'Divorced']) &
    (df['Income'] >= 90000)
]
display(ex3)

,ID,Refund,Marital Status,Income,Cheat
0,1,Yes,Single,"125,000.00",No
4,5,No,Divorced,"95,000.00",Yes
6,7,Yes,Divorced,"220,000.00",No
9,10,No,Single,"90,000.00",Yes
10,11,No,Divorced,"95,000.00",Yes
15,16,No,Divorced,"95,000.00",Yes
17,7,Yes,Divorced,"220,000.00",No


In [91]:
# Exercise 4
ex4 = df[
    (df['Refund'] == 'Yes') &
    (df['Cheat'] == 'No') &
    (df['Income'].notna())
]
display(ex4)

,ID,Refund,Marital Status,Income,Cheat
0,1,Yes,Single,"125,000.00",No
3,4,Yes,Married,"120,000.00",No
6,7,Yes,Divorced,"220,000.00",No
17,7,Yes,Divorced,"220,000.00",No


In [92]:
# Exercise 5
ex5 = (
    df[['ID', 'Marital Status', 'Income', 'Refund', 'Cheat']]
      .sort_values('Income', ascending=False)
      .head(5)
)
display(ex5)

,ID,Marital Status,Income,Refund,Cheat
13,14,Married,"1,000,000.00",No,No
6,7,Divorced,"220,000.00",Yes,No
17,7,Divorced,"220,000.00",Yes,No
0,1,Single,"125,000.00",Yes,No
3,4,Married,"120,000.00",Yes,No


In [93]:
# Exercise 6
ex6 = df.copy()
ex6['income_band'] = pd.cut(
    ex6['Income'],
    bins=[-np.inf, 80000, 120000, np.inf],
    right=False,
    labels=['Low', 'Medium', 'High']
)
display(ex6[['ID', 'Income', 'income_band']])

,ID,Income,income_band
0,1,"125,000.00",High
1,2,"100,000.00",Medium
2,3,"70,000.00",Low
3,4,"120,000.00",High
4,5,"95,000.00",Medium
5,6,"60,000.00",Low
6,7,"220,000.00",High
7,8,"85,000.00",Medium
8,9,"75,000.00",Low
9,10,"90,000.00",Medium


In [94]:
# Exercise 7
ex7 = (
    df.groupby('Marital Status')
      .agg(
          record_count=('ID', 'count'),
          mean_income=('Income', 'mean'),
          median_income=('Income', 'median'),
          cheat_yes=('Cheat', lambda s: (s == 'Yes').sum())
      )
)
display(ex7)

,record_count,mean_income,median_income,cheat_yes
Marital Status,,,,
Divorced,5,"145,000.00","95,000.00",3
Married,8,"235,833.33","87,500.00",0
Single,5,"88,000.00","85,000.00",2


In [95]:
# Exercise 8
ex8 = df.query('75000 <= Income <= 125000')
display(ex8)

,ID,Refund,Marital Status,Income,Cheat
0,1,Yes,Single,"125,000.00",No
1,2,No,Married,"100,000.00",No
3,4,Yes,Married,"120,000.00",No
4,5,No,Divorced,"95,000.00",Yes
7,8,No,Single,"85,000.00",Yes
8,9,No,Married,"75,000.00",No
9,10,No,Single,"90,000.00",Yes
10,11,No,Divorced,"95,000.00",Yes
15,16,No,Divorced,"95,000.00",Yes


In [96]:
# Exercise 9
ex9 = df.copy().set_index('ID')
ex9 = ex9.drop(index=14).reset_index()
print('ID 14 present?', (ex9['ID'] == 14).any())
display(ex9.head())

ID 14 present? False


,ID,Refund,Marital Status,Income,Cheat
0,1,Yes,Single,"125,000.00",No
1,2,No,Married,"100,000.00",No
2,3,No,Single,"70,000.00",No
3,4,Yes,Married,"120,000.00",No
4,5,No,Divorced,"95,000.00",Yes


# Laboratory Summary

This laboratory implemented the lecture's EDA concepts using `taxes_unprocessed_with_duplicates.csv`:

- built and inspected a DataFrame;
- calculated aggregate statistics;
- detected and handled missing values;
- distinguished exact duplicates from analytically meaningful duplicates;
- used `drop()`, `replace()`, `rename()`, `astype()`, and numeric conversion;
- selected and sliced with `loc[]`, `iloc[]`, and standard slicing;
- filtered using Boolean masks, `isin()`, `between()`, and `query()`;
- normalized income values;
- sorted records and selected distinct values;
- created and conditionally modified columns;
- applied `describe()`, maximum-value filtering, and index manipulation;
- implemented split–apply–combine with `groupby()`;
- created named group-level aggregations;
- audited data quality and inspected an influential outlier;

The central lesson is that **EDA is not a single command**. It is a sequence of justified, verifiable decisions that transform raw data into trustworthy analytical evidence.